# Session Manager

Pilotage des sessions sans network volume. Les scripts vivent sur HF (`rafatribe/Personnal_Loras/session/`), ce notebook ne fait que les lancer.
Deux envs : **image** (Z-Image / Krea, `my_env`) et **video** (ARP / LTX, `venv_cu130`). Seule la cellule 1 demande de choisir, les autres detectent l'env toutes seules.

**Routine image**
1. Cellule **1** avec `SESSION_ENV="image"`.
2. Lancer ComfyUI avec la **derniere cellule** de `comfyui_runpod_Z-Image.ipynb`.
3. Cellule **4** : zipper les outputs, les telecharger, Terminate.

**Routine video (ARP)**
1. Cellule **1** avec `SESSION_ENV="video"` (Container Disk 200 Go, port HTTP 8765 expose).
2. Lancer la **cellule 7** de `arp_runpod_5090_cu130.ipynb`.
3. Recuperer les rendus (fleche de telechargement dans ARP), puis Terminate. Chaque pod video repart d'un disque propre.

Sur un pod d'origine (avec network volume), la cellule 1 refuse de tourner : l'env y est deja.


## 1. Boot de la session (env + modeles)
Choisis l'env et les profils de modeles dans les premieres lignes. Un modele absent reste telechargeable a la demande via le Model Manager.


In [ ]:
%%bash
export SESSION_ENV="image"   # image (Z-Image / Krea) ou video (ARP / LTX)
# Profils separes par des virgules. Vide = defaut (krea2 en image, arp en video).
# image : krea2 | "tout" = tous les modeles de l'ancien volume image (base toujours incluse)
# video : arp
export PROFILS=""

if [ -f /workspace/ComfyUI/main.py ] && [ ! -L /workspace/ComfyUI ] && [ ! -f /workspace/.env_restored ]; then
  echo "Pod d'origine detecte (ComfyUI deja present dans /workspace) : boot inutile."
  exit 0
fi
curl -sL -H "Authorization: Bearer $HF_TOKEN" https://huggingface.co/rafatribe/Personnal_Loras/resolve/main/session/boot.sh | bash


## 2. Suivi du telechargement des modeles
A relancer autant de fois que tu veux pendant le boot.

In [ ]:
%%bash
LOG=/local/fetch_models.log
[ -f "$LOG" ] || LOG=/workspace/fetch_models.log
echo "$(grep -c '^\[' "$LOG" 2>/dev/null || echo 0) fichiers traites"
tail -n 15 "$LOG" 2>/dev/null
df -h / | tail -1 | awk '{print "Container Disk : " $4 " libres sur " $2}'

## 3. Arreter ComfyUI
A faire avant un snapshot (cellule 6).

In [ ]:
%%bash
pkill -f arp_launch.py && echo "Interface ARP arretee."
if pkill -f "main.py --listen"; then sleep 3; echo "ComfyUI arrete."; else echo "ComfyUI ne tournait pas."; fi


## 4. Zipper les outputs de la session
Le zip apparait dans `/workspace`, dans le panneau de gauche de Jupyter : clic droit > Download.

En video, les rendus ARP sont dans `ai-remaster-pipeline/output/reassembled/` : telecharge-les plutot avec la fleche de l'interface ARP.


In [ ]:
%%bash
cd /workspace/ComfyUI/output || exit 1
NAME="outputs_$(date +%Y%m%d-%H%M).zip"
if [ -d /local ]; then
  python3 -m zipfile -c "/local/$NAME" . && ln -sf "/local/$NAME" "/workspace/$NAME"
else
  python3 -m zipfile -c "/workspace/$NAME" .
fi
ls -lhL "/workspace/$NAME"

## 5. Mettre a jour ComfyUI et les custom nodes

Les dependances sont installees dans le venv de l'env (`my_env` ou `venv_cu130`) (avec tes versions epinglees). Les nodes que tu as modifies localement, et les nodes ARP (copies sans git), sont laisses tels quels.

Ensuite : relance ComfyUI avec la derniere cellule de ton notebook (elle reapplique les hotfixes) et teste tes workflows avant tout snapshot.

In [ ]:
%%bash
if [ -e /workspace/venv_cu130 ]; then export SESSION_ENV=video VENV=venv_cu130; else export SESSION_ENV=image VENV=my_env; fi
PIP=/workspace/$VENV/bin/pip
echo "Env : $SESSION_ENV"
cd /workspace/ComfyUI || exit 1

echo "=== ComfyUI ==="
DELETED=$(git ls-files --deleted)
[ -n "$DELETED" ] && git checkout -q -- $DELETED
if [ -n "$(git status --porcelain --untracked-files=no)" ]; then
  echo "!! ComfyUI a des modifications locales, mise a jour sautee :"
  git status --short --untracked-files=no
else
  B=$(git rev-parse --short HEAD)
  git fetch -q origin && git checkout -q master && git pull -q --ff-only origin master
  A=$(git rev-parse --short HEAD)
  echo "ComfyUI : $B -> $A"
  [ "$B" != "$A" ] && $PIP install -q -r requirements.txt
fi

echo ""
echo "=== Custom nodes ==="
for d in custom_nodes/*/; do
  n=$(basename "$d")
  [ -d "$d/.git" ] || { echo "  -- $n : pas un depot git, ignore"; continue; }
  if [ -n "$(git -C "$d" status --porcelain --untracked-files=no)" ]; then
    echo "  !! $n : modifie localement, laisse tel quel"; continue
  fi
  B=$(git -C "$d" rev-parse --short HEAD)
  git -C "$d" pull -q --ff-only 2>/dev/null || { echo "  !! $n : mise a jour impossible"; continue; }
  A=$(git -C "$d" rev-parse --short HEAD)
  if [ "$B" != "$A" ]; then
    echo "  OK $n : $B -> $A"
    [ -f "$d/requirements.txt" ] && $PIP install -q -r "$d/requirements.txt"
  fi
done
echo ""
echo "Termine. Relance ComfyUI avec la derniere cellule de ton notebook, puis teste."

## 6. Snapshot de l'env (nouvelle base pour les prochains pods)

Uniquement apres avoir teste tes workflows. Arrete ComfyUI avant (cellule 3). L'ancien snapshot reste actif tant que le nouveau n'est pas entierement envoye.

In [ ]:
%%bash
if [ -e /workspace/venv_cu130 ]; then export SESSION_ENV=video VENV=venv_cu130; else export SESSION_ENV=image VENV=my_env; fi
export PATH=/workspace/$VENV/bin:$PATH
pgrep -f "main.py --listen" > /dev/null && { echo "ComfyUI tourne encore : lance d'abord la cellule 3."; exit 1; }
curl -sL -H "Authorization: Bearer $HF_TOKEN" https://huggingface.co/rafatribe/Personnal_Loras/resolve/main/session/snapshot_env.sh | bash


## 7. Garder les modeles ajoutes pendant la session (optionnel)

Si tu as telecharge de nouveaux modeles (Model Manager) et que tu veux les retrouver au prochain boot : met a jour la liste des modeles de session a partir de ce qui est sur le disque.

In [ ]:
%%bash
if [ -e /workspace/venv_cu130 ]; then export SESSION_ENV=video VENV=venv_cu130; else export SESSION_ENV=image VENV=my_env; fi
export PATH=/workspace/$VENV/bin:$PATH
LISTE=session_models.txt; [ "$SESSION_ENV" = video ] && LISTE=session_models_video.txt
cd /workspace/ComfyUI/models || exit 1
find -L . -type f \( -name "*.safetensors" -o -name "*.ckpt" -o -name "*.pt" -o -name "*.pth" -o -name "*.gguf" -o -name "*.bin" \) -size +0 \
  | sed 's#^\./##' | grep -v '^sam3/sam3.pt$' | grep -v '/\.cache/' | sort > /tmp/session_models.txt
echo "$(wc -l < /tmp/session_models.txt) modeles dans la liste $LISTE (profil \"tout\")"
hf upload rafatribe/Personnal_Loras /tmp/session_models.txt "session/$LISTE" --commit-message "Liste modeles $(date +%F)"